In [ ]:
import re
import json
import statistics
from collections import defaultdict
from dataclasses import dataclass, field, asdict

## Categorizar Log

In [ ]:
# Linha base do log4j do Spark:
# 26/07/27 09:48:01 INFO DAGScheduler: mensagem...
LINE_RE = re.compile(
    r"^(?P<ts>\d{2}/\d{2}/\d{2} \d{2}:\d{2}:\d{2}) "
    r"(?P<level>INFO|WARN|ERROR|DEBUG) "
    r"(?P<component>[\w$.]+): "
    r"(?P<msg>.*)$"
)

PADROES_DE_EVENTOS = {
    "job_start": re.compile(r"^Got job (?P<job_id>\d+) \((?P<action>.*?)\) with (?P<partitions>\d+) output partitions$"),
    "job_finish": re.compile(r"^Job (?P<job_id>\d+) finished: .*?, took (?P<duration_s>[\d.]+) s$"),
    "stage_submit": re.compile(r"^Submitting (?P<stage>ResultStage|ShuffleMapStage) (?P<stage_id>\d+)"),
    "stage_finish": re.compile(r"^(?P<stage>ResultStage|ShuffleMapStage) (?P<stage_id>\d+) \(.*?\) finished in (?P<duration_s>[\d.]+) s$"),
    "task_start": re.compile(r"^Starting task (?P<task_id>[\d.]+) in stage (?P<stage_id>[\d.]+) \(TID (?P<tid>\d+)\).*?executor (?P<executor>\d+)"),
    "task_finish": re.compile(r"^Finished task (?P<task_id>[\d.]+) in stage (?P<stage_id>[\d.]+) \(TID (?P<tid>\d+)\) in (?P<duration_ms>\d+) ms on .*? \(executor (?P<executor>\d+)\)"),
}


@dataclass
class LogEvent:
    ts: str
    level: str
    component: str
    event_type: str
    data: dict = field(default_factory=dict)


def parse_log(text: str) -> list[LogEvent]:
    events = []
    for raw_line in text.splitlines():
        m = LINE_RE.match(raw_line.strip())
        if not m:
            continue  # linha de stack trace / continuação; tratar à parte se precisar

        ts, level, component, msg = m.group("ts", "level", "component", "msg")

        matched = False
        for event_type, pattern in PADROES_DE_EVENTOS.items():
            em = pattern.match(msg)
            if em:
                events.append(LogEvent(ts, level, component, event_type, em.groupdict()))
                matched = True
                break

        if not matched and level in ("WARN", "ERROR"):
            # warnings/erros sem padrão específico ainda: guarda cru pra revisão manual
            events.append(LogEvent(ts, level, component, "raw_warn_error", {"msg": msg}))

    return events


def aggregate_by_stage(events: list[LogEvent]) -> list[dict]:
    """Junta task_start+task_finish por TID e agrega por stage_id,
    devolvendo estatísticas em vez de uma linha por task."""

    # tid -> {executor, task_id, stage_id, duration_ms}
    tasks: dict[str, dict] = {}
    for e in events:
        if e.event_type == "task_start":
            tasks.setdefault(e.data["tid"], {}).update(e.data)
        elif e.event_type == "task_finish":
            tasks.setdefault(e.data["tid"], {}).update(e.data)

    # agrupa tasks completas (com duration_ms) por stage_id
    # obs: nas tasks o stage_id vem como "stageId.attemptId" (ex: "0.0");
    # normalizamos pra "stageId" puro pra casar com stage_submit/stage_finish
    by_stage: dict[str, list[dict]] = defaultdict(list)
    for t in tasks.values():
        if "duration_ms" in t and "stage_id" in t:
            stage_id = t["stage_id"].split(".")[0]
            by_stage[stage_id].append(t)

    # duração total do stage, vinda do evento stage_finish
    stage_meta = {}
    for e in events:
        if e.event_type == "stage_finish":
            stage_meta[e.data["stage_id"]] = {"duration_s": float(e.data["duration_s"])}

    summary = []
    for stage_id, task_list in by_stage.items():
        durations = [int(t["duration_ms"]) for t in task_list]
        per_executor = defaultdict(list)
        for t in task_list:
            per_executor[t.get("executor", "?")].append(int(t["duration_ms"]))

        durations_sorted = sorted(durations)
        p95_idx = max(0, int(len(durations_sorted) * 0.95) - 1)
        median = statistics.median(durations)

        summary.append({
            "stage_id": stage_id,
            "num_tasks": len(task_list),
            "duration_s": stage_meta.get(stage_id, {}).get("duration_s"),
            "task_duration_ms": {
                "min": min(durations),
                "max": max(durations),
                "mean": round(statistics.mean(durations), 1),
                "median": median,
                "p95": durations_sorted[p95_idx],
            },
            "tasks_per_executor": {ex: len(v) for ex, v in per_executor.items()},
            "executor_duration_ms": {
                ex: {"mean": round(statistics.mean(v), 1), "total": sum(v)}
                for ex, v in per_executor.items()
            },
            "skew_ratio": round(max(durations) / median, 2) if median > 0 else None,
        })

    return sorted(summary, key=lambda s: int(s["stage_id"]))


if __name__ == "__main__":
    import sys

    path = (r'C:\Users\Pichau\Documents\spark_log_reader\log_REF_TRACKING_sucesso.txt')
    with open(path, encoding="utf-8") as f:
        content = f.read()

    parsed = parse_log(content)

    stage_summary = aggregate_by_stage(parsed)
    rare_events = [asdict(e) for e in parsed if e.event_type == "raw_warn_error"]

    output = {
        "stages": stage_summary,
        "raw_warn_error": rare_events,
    }
    print(json.dumps(output, indent=2, ensure_ascii=False))
    open("log_agregado.json", "w", encoding="utf-8").write(json.dumps(output, indent=2, ensure_ascii=False))
    